# Phase 2 — ORACLE causal-capacity sur seed42 (réel ACCESS-CM2)

**Question méta** : *Le DAG seed42 6-node + Stage 1 frozen contient-il assez d'information pour battre noncausal v4 sur F1@p99 dans le scénario IDÉAL (gating oracle parfait per-pixel) ?*

Si OUI → le bottleneck est l'architecture/loss (commit CDPM-min sur seed42, Phase 3, ~25-30h).
Si NON → le bottleneck est le DAG content (extension 9 nodes IVT+CAPE obligatoire, ~17h + retrain Stage 1).

**Design (Agent critique a36fca58)** : *« Take noncausal CorrDiff v4 as Path 2, freeze it. Feed cheating Path 1 = best DAG-conformant approximation of HR_true. Gate-fuse per pixel. Compute F1@p99 ; if ≤ 0.550 → no architecture can win, if > 0.580 → causal signal extractable. »*

**Variantes d'oracle testées** :
1. **`oracle_perfect_gate`** : pour chaque pixel, choisir entre `(baseline + μ_HR)` et `v4_pred` la valeur la plus proche de HR_true. Borne supérieure stricte.
2. **`oracle_optimal_alpha`** : per-pixel `α*(HR_true)` = projection L2 optimale entre les 2 chemins.
3. **`oracle_scaled_mu_HR`** : `α* · μ_HR(LR) + baseline` avec scaling per-sample. Teste si la direction rank-5 du DAG est informative même si la magnitude est mal calibrée.

**Compute attendu** : ~30-45 min A100 (K9 val ≈ 700 samples × 18 steps Heun pour noncausal v4 inference, le reste est trivial).

**Décision automatique** :
- `F1@p99(oracle_perfect_gate) > 0.580` → GO Phase 3 (CDPM-min sur seed42, ~25-30h)
- `F1@p99 ≤ 0.550` → STOP causal-stage2 ; pivoter vers extension DAG 9-node (Stage 1 retrain)
- `F1@p99 ∈ (0.550, 0.580)` → marginal, décision utilisateur

Baseline à battre : **noncausal v4 F1@p99 = 0.512**.

In [ ]:
# === Cell 1 : Colab bootstrap (Drive mount + repo clone + deps) ===
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = Path('/content/drive/MyDrive/climate_data')
    REPO_DIR = Path('/content/stcdgm')
    if not REPO_DIR.exists():
        os.system(f'git clone https://github.com/leonelkenfack/stcdgm.git {REPO_DIR}')
    os.system(f'cd {REPO_DIR} && git fetch && git checkout four-node-causal && git pull')
    sys.path.insert(0, str(REPO_DIR))
    os.system('pip install -q einops scipy h5py netCDF4 xarray dask zarr')
else:
    DRIVE_ROOT = Path('c:/Users/reall/Desktop/climate_data')
    REPO_DIR = DRIVE_ROOT
    sys.path.insert(0, str(REPO_DIR))

print(f'Drive root : {DRIVE_ROOT}')
print(f'Repo dir   : {REPO_DIR}')

In [ ]:
# === Cell 2 : Imports + paths + config ===
import math
import time
import json
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

# === Paths (extracted from st_cdgm_seed42_eval.ipynb) ===
STAGE1_CKPT = DRIVE_ROOT / 'oracle_full' / 'seed_42' / 'epoch_last.pth'
NONCAUSAL_DIR = DRIVE_ROOT / 'ckpt_v2_corrdiff_normal'
NONCAUSAL_METRICS = NONCAUSAL_DIR / 'final_validation_metrics.json'
OUTPUT_DIR = DRIVE_ROOT / 'oracle_full' / 'seed_42' / 'phase2_oracle'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# === K9 split (extracted) ===
K9_DATES = {'val': ['2010-01-01', '2011-12-31']}

# === Decision thresholds ===
F1_PASS = 0.580
F1_FAIL = 0.550
F1_NONCAUSAL_V4 = 0.5123          # confirmed in audit
RMSE_NONCAUSAL_V4 = 0.1243
PEARSON_NONCAUSAL_V4 = 0.8344

for p in [STAGE1_CKPT, NONCAUSAL_DIR]:
    print(f'  {"✓" if p.exists() else "✗"} {p}')

# Load noncausal v4 baseline metrics for sanity
if NONCAUSAL_METRICS.exists():
    with open(NONCAUSAL_METRICS) as f:
        v4_metrics = json.load(f)
    print(f'\nNoncausal v4 metrics loaded :')
    for k in ['rmse', 'pearson_corr', 'f1_extremes']:
        v = v4_metrics.get(k)
        if v is not None:
            print(f'  {k:>20} : {v}')

In [ ]:
# === Cell 3 : Load K9 val dataset ===
# Uses the existing st_cdgm.data.pipeline infrastructure
from st_cdgm.config import CONFIG
from st_cdgm.data import pipeline

# Override K9 dates if needed
SEQ_LEN = int(CONFIG.data.seq_len)
STRIDE = int(CONFIG.data.stride)

# Build the validation sequence dataset
val_dataset = pipeline.build_sequence_dataset(
    split='val',
    seq_len=SEQ_LEN,
    stride=STRIDE,
    as_torch=True,
)
print(f'K9 val dataset    : {len(val_dataset)} samples')
print(f'seq_len           : {SEQ_LEN}')
print(f'stride            : {STRIDE}')

# Quick sanity : peek 1 sample
_sample = val_dataset[0]
for k, v in (_sample.items() if hasattr(_sample, 'items') else []):
    if hasattr(v, 'shape'):
        print(f'  {k:20s} shape={tuple(v.shape)} dtype={v.dtype}')

In [ ]:
# === Cell 4 : Load frozen Stage 1 seed42 (Q_phys=0.998) ===
from st_cdgm.models.encoder import GNNEncoder
from st_cdgm.models.regression import RegressionHead
from st_cdgm.models.rcn_cell import RCNCell
from st_cdgm.training.two_stage import freeze_stage1

# Build the Stage 1 components
encoder = GNNEncoder(
    n_nodes=int(CONFIG.model.n_nodes),
    n_input_features=int(CONFIG.model.n_input_features),
    hidden_dim=int(CONFIG.model.encoder_hidden_dim),
    n_layers=int(CONFIG.model.encoder_n_layers),
).to(DEVICE)

rcn_cell = RCNCell(
    n_nodes=int(CONFIG.model.n_nodes),
    hidden_dim=int(CONFIG.model.encoder_hidden_dim),
).to(DEVICE)

regression_head = RegressionHead(
    n_nodes=int(CONFIG.model.n_nodes),
    hidden_dim=int(CONFIG.model.encoder_hidden_dim),
    out_spatial=(int(CONFIG.data.target_h), int(CONFIG.data.target_w)),
).to(DEVICE)

# Load checkpoint
ck = torch.load(STAGE1_CKPT, map_location=DEVICE, weights_only=False)
print(f'Checkpoint keys : {list(ck.keys())[:8]}...')

def _load_sd(model, sd):
    if sd is None:
        print(f'  ⚠ {model.__class__.__name__} state_dict missing')
        return
    out = model.load_state_dict(sd, strict=False)
    print(f'  {model.__class__.__name__} : missing {len(out.missing_keys)} unexpected {len(out.unexpected_keys)}')

_load_sd(encoder, ck.get('encoder_state_dict'))
_load_sd(rcn_cell, ck.get('rcn_cell_state_dict'))
_load_sd(regression_head, ck.get('regression_head_state_dict'))

# Extract A_dag
A_dag = ck.get('A_dag') or ck.get('rcn_cell_state_dict', {}).get('A_dag')
if A_dag is not None:
    A_dag = A_dag.to(DEVICE)
    print(f'\nA_dag shape : {tuple(A_dag.shape)}')
    print(f'A_dag rank  : {torch.linalg.matrix_rank(A_dag).item()}')
    print(f'A_dag norm  : {A_dag.norm().item():.4f}')

# Freeze
freeze_stage1(encoder, rcn_cell, regression_head)
for m in [encoder, rcn_cell, regression_head]:
    m.eval()
print('\nStage 1 frozen & in eval mode.')

In [ ]:
# === Cell 5 : Load noncausal v4 (ckpt_v2_corrdiff_normal) ===
# Reusing whatever infrastructure was used in st_cdgm_seed42_eval.ipynb
from st_cdgm.models.diffusion_v2_core_eval import DiffusionV2CoreEval

noncausal_v4 = DiffusionV2CoreEval(CONFIG).to(DEVICE)

# Find ckpt file (the dir contains multiple ckpts; pick the latest val-best one)
import glob
candidates = sorted(glob.glob(str(NONCAUSAL_DIR / '*.pth')))
if not candidates:
    raise FileNotFoundError(f'No .pth in {NONCAUSAL_DIR}')
v4_ckpt_path = NONCAUSAL_DIR / 'val_best.pth'
if not v4_ckpt_path.exists():
    v4_ckpt_path = Path(candidates[-1])
print(f'Noncausal v4 ckpt : {v4_ckpt_path}')

v4_ck = torch.load(v4_ckpt_path, map_location=DEVICE, weights_only=False)
v4_sd = v4_ck.get('diffusion_ema_state_dict') or v4_ck.get('diffusion_state_dict') or v4_ck
_load_sd(noncausal_v4, v4_sd)
for p in noncausal_v4.parameters():
    p.requires_grad_(False)
noncausal_v4.eval()
print('Noncausal v4 frozen & in eval mode.')

In [ ]:
# === Cell 6 : Inference helpers ===
from st_cdgm.training.two_stage import precompute_stage1_outputs

@torch.no_grad()
def stage1_inference(loader):
    """Compute μ_HR(LR), baseline_log, valid_mask on the loader.
    Returns dict of stacked tensors on CPU."""
    cache = precompute_stage1_outputs(
        loader, encoder, rcn_cell, regression_head,
        config=CONFIG, device=DEVICE,
    )
    # cache must contain: mu_HR, baseline_log, valid_mask, delta_target (= HR - baseline - μ_HR)
    return cache

@torch.no_grad()
def noncausal_v4_inference(loader, n_steps=18):
    """Run noncausal v4 EDM Heun sampler on K9 val. Returns [N, 1, H, W] predictions."""
    preds, HRs, masks = [], [], []
    t0 = time.time()
    for i, batch in enumerate(loader):
        cond = batch['conditioning'].to(DEVICE) if 'conditioning' in batch else batch['LR'].to(DEVICE)
        baseline_log = batch['baseline_log'].to(DEVICE)
        # μ_HR is ZEROED in Mardani fix (and not used by noncausal v4 anyway)
        mu_HR_zero = torch.zeros_like(baseline_log)
        # Sample
        sample = noncausal_v4.sample(
            conditioning=cond, mu_HR=mu_HR_zero, baseline_log=baseline_log,
            num_steps=n_steps, scheduler_type='edm_karras',
        )
        preds.append(sample.cpu())
        if 'HR' in batch:
            HRs.append(batch['HR'].cpu())
        if 'valid_mask' in batch:
            masks.append(batch['valid_mask'].cpu())
        if (i + 1) % 20 == 0:
            elapsed = time.time() - t0
            eta = elapsed / (i + 1) * (len(loader) - i - 1)
            print(f'  batch {i+1}/{len(loader)}  elapsed={elapsed:.0f}s  ETA={eta:.0f}s')
    return {
        'v4_pred': torch.cat(preds, dim=0),
        'HR_true': torch.cat(HRs, dim=0) if HRs else None,
        'valid_mask': torch.cat(masks, dim=0) if masks else None,
    }

print('Inference helpers defined.')

In [ ]:
# === Cell 7 : Run inference on K9 val ===
from torch.utils.data import DataLoader

BATCH_SIZE = 8
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Stage 1
print('=== Stage 1 inference ===')
t0 = time.time()
stage1_cache = stage1_inference(val_loader)
print(f'  done in {time.time()-t0:.0f}s')
print(f'  μ_HR shape  : {tuple(stage1_cache["mu_HR"].shape)}')
print(f'  baseline    : {tuple(stage1_cache["baseline_log"].shape)}')
if 'valid_mask' in stage1_cache:
    print(f'  valid_mask  : {tuple(stage1_cache["valid_mask"].shape)}')

# Build a loader that ALSO carries HR_true (for oracle gating)
# Note: stage1_cache may already have delta_target = HR - baseline - μ_HR, so we can reconstruct HR_true
HR_true = stage1_cache['baseline_log'] + stage1_cache['mu_HR'] + stage1_cache['delta_target']
print(f'  HR_true reconstructed shape : {tuple(HR_true.shape)}')

# Noncausal v4
print()
print('=== Noncausal v4 inference (18 Heun steps) ===')
t0 = time.time()
v4_out = noncausal_v4_inference(val_loader, n_steps=18)
print(f'  done in {time.time()-t0:.0f}s')
print(f'  v4_pred shape : {tuple(v4_out["v4_pred"].shape)}')

In [ ]:
# === Cell 8 : Compute oracle variants ===

mu_HR = stage1_cache['mu_HR']          # [N, 1, H, W]
baseline = stage1_cache['baseline_log']
v4_pred = v4_out['v4_pred']
if v4_out['HR_true'] is not None:
    HR_true = v4_out['HR_true']
valid_mask = stage1_cache.get('valid_mask', torch.ones_like(HR_true))

# Path 1 candidate : (baseline + μ_HR) — the 'DAG-conformant' prediction
path1 = baseline + mu_HR
# Path 2 candidate : noncausal v4 prediction
path2 = v4_pred

# Sanity
print(f'Path 1 (baseline + μ_HR)  range : [{path1.min():.3f}, {path1.max():.3f}]')
print(f'Path 2 (v4 pred)           range : [{path2.min():.3f}, {path2.max():.3f}]')
print(f'HR_true                    range : [{HR_true.min():.3f}, {HR_true.max():.3f}]')
print(f'valid_mask coverage              : {valid_mask.float().mean().item():.4f}')

# === Oracle 1 : perfect binary gate per pixel ===
# For each pixel : pick path1 if closer to HR_true, else path2
err1 = (path1 - HR_true).abs()
err2 = (path2 - HR_true).abs()
gate_binary = (err1 < err2).float()                # 1 = use path1, 0 = use path2
oracle_perfect = gate_binary * path1 + (1 - gate_binary) * path2
frac_path1 = gate_binary.mean().item()
print(f'\nOracle 1 (binary gate) : path1 selected on {frac_path1*100:.1f}% of pixels')

# === Oracle 2 : optimal continuous α per pixel ===
# α* = ((p2 - HR) · (p2 - p1)) / ‖p1 - p2‖² , clamped to [0, 1]
diff = path1 - path2
denom = diff.pow(2).clamp_min(1e-8)
alpha_opt = ((path2 - HR_true) * (path2 - path1)) / denom
alpha_opt = alpha_opt.clamp(0.0, 1.0)
oracle_alpha = alpha_opt * path1 + (1 - alpha_opt) * path2
print(f'Oracle 2 (continuous α) : mean α* = {alpha_opt.mean().item():.4f}')

# === Oracle 3 : scaled μ_HR (per-sample best magnitude) ===
# For each sample : find β* such that (baseline + β*·μ_HR) is closest to HR_true
# β* = <μ_HR, HR_true - baseline> / ‖μ_HR‖²  per sample
N = HR_true.shape[0]
mu_flat = mu_HR.view(N, -1)
tgt_flat = (HR_true - baseline).view(N, -1)
beta_opt = (mu_flat * tgt_flat).sum(dim=1) / (mu_flat.pow(2).sum(dim=1).clamp_min(1e-8))
beta_opt = beta_opt.clamp(0.5, 2.0)               # avoid extreme scaling
print(f'Oracle 3 (scaled μ_HR) : β* mean={beta_opt.mean():.3f}, std={beta_opt.std():.3f}')
oracle_scaled = baseline + beta_opt.view(N, 1, 1, 1) * mu_HR

# === Combine oracle 3 with v4 (per-sample) ===
# Gate between (baseline + β*·μ_HR) and v4 — per pixel binary
err_scaled = (oracle_scaled - HR_true).abs()
err_v4 = (path2 - HR_true).abs()
gate3 = (err_scaled < err_v4).float()
oracle_combined = gate3 * oracle_scaled + (1 - gate3) * path2
print(f'Oracle 3+v4 (combined) : scaled-μ_HR selected on {gate3.mean()*100:.1f}% of pixels')

In [ ]:
# === Cell 9 : Compute metrics for each variant ===
from st_cdgm.evaluation import compute_f1_extremes, compute_spectrum_distance

def _pearson(a, b, eps=1e-8):
    a_c = a - a.mean()
    b_c = b - b.mean()
    return (a_c * b_c).sum().item() / (a_c.norm().item() * b_c.norm().item() + eps)

def compute_metrics(pred, true, mask):
    pred_v = pred * mask
    true_v = true * mask
    valid_n = mask.sum().clamp_min(1.0)
    rmse = ((pred_v - true_v).pow(2).sum() / valid_n).sqrt().item()
    mae = (pred_v - true_v).abs().sum().item() / valid_n.item()
    flat_p = pred_v.flatten()
    flat_t = true_v.flatten()
    pearson = _pearson(flat_p, flat_t)
    f1 = compute_f1_extremes(pred.numpy(), true.numpy(), threshold_percentiles=[95.0, 99.0])
    return {
        'rmse': rmse,
        'mae': mae,
        'pearson': pearson,
        'f1_p95': float(f1.get('p95', 0.0)),
        'f1_p99': float(f1.get('p99', 0.0)),
    }

print('=== Computing metrics per variant ===')
variants = {
    'baseline_only': baseline,
    'baseline_plus_mu_HR': path1,
    'noncausal_v4': path2,
    'oracle_perfect_gate': oracle_perfect,
    'oracle_optimal_alpha': oracle_alpha,
    'oracle_scaled_mu_HR': oracle_scaled,
    'oracle_combined_scaled_v4': oracle_combined,
}

results = {}
for name, pred in variants.items():
    m = compute_metrics(pred, HR_true, valid_mask)
    results[name] = m
    print(f'  {name:30s} RMSE={m["rmse"]:.4f}  Pearson={m["pearson"]:.4f}  F1@p99={m["f1_p99"]:.4f}  F1@p95={m["f1_p95"]:.4f}')

In [ ]:
# === Cell 10 : Decision tree + save ===

print('=' * 80)
print('DÉCISION ORACLE')
print('=' * 80)
print()
f1_oracle_perfect = results['oracle_perfect_gate']['f1_p99']
f1_oracle_alpha = results['oracle_optimal_alpha']['f1_p99']
f1_noncausal = results['noncausal_v4']['f1_p99']

print(f'  F1@p99 noncausal v4 (réf)      : {f1_noncausal:.4f}')
print(f'  F1@p99 oracle perfect gate     : {f1_oracle_perfect:.4f}  (Δ = {f1_oracle_perfect - f1_noncausal:+.4f})')
print(f'  F1@p99 oracle optimal α        : {f1_oracle_alpha:.4f}  (Δ = {f1_oracle_alpha - f1_noncausal:+.4f})')
print(f'  F1@p99 oracle scaled μ_HR + v4 : {results["oracle_combined_scaled_v4"]["f1_p99"]:.4f}')
print()
print(f'  Seuils décision :')
print(f'    PASS si F1@p99(oracle_perfect) > {F1_PASS}')
print(f'    FAIL si F1@p99(oracle_perfect) ≤ {F1_FAIL}')
print()

if f1_oracle_perfect > F1_PASS:
    verdict = 'PASS_ORACLE'
    print('  ✓ VERDICT : PASS_ORACLE')
    print('  ▶ Le DAG seed42 6-node CONTIENT assez d\'info pour battre v4 sur extrêmes.')
    print('  ▶ Le bottleneck est l\'architecture / la loss, PAS le DAG content.')
    print('  ▶ NEXT : Phase 3 = commit CDPM-min sur seed42 (pinball multi-quantile + decoupled, ~25-30h).')
    print('  ▶ DAG extension 9-node devient OPTIONNEL (peut-être plus tard pour gains additionnels).')
elif f1_oracle_perfect <= F1_FAIL:
    verdict = 'FAIL_DAG_LIMITED'
    print('  ✗ VERDICT : FAIL_DAG_LIMITED')
    print('  ▶ Même avec gating ORACLE, le DAG seed42 ne fournit pas assez d\'info.')
    print('  ▶ Le bottleneck EST le DAG content (probable absence de q_850, ω_500).')
    print('  ▶ NEXT : Phase 3 = extension DAG 9-node + Stage 1 retrain (~17h)')
    print('  ▶ AVANT de commit pinball, parce que pinball + bottleneck DAG = même résultat.')
else:
    verdict = 'MARGINAL'
    print(f'  ◯ VERDICT : MARGINAL (F1@p99 ∈ [{F1_FAIL}, {F1_PASS}])')
    print('  ▶ Gain réel mais petit → Pareto frontier Q_phys/skill confirmée.')
    print('  ▶ Décision utilisateur : commit pinball (~25h) pour grappiller, OU extension DAG (~17h) pour potentiel plus grand.')

# Save full results
save_path = OUTPUT_DIR / 'oracle_metrics.json'
save = {
    'verdict': verdict,
    'thresholds': {'pass': F1_PASS, 'fail': F1_FAIL},
    'noncausal_v4_baseline': {
        'rmse': RMSE_NONCAUSAL_V4,
        'pearson': PEARSON_NONCAUSAL_V4,
        'f1_p99': F1_NONCAUSAL_V4,
    },
    'variants': results,
    'oracle_perfect_gate_fraction_path1': frac_path1,
    'oracle_alpha_mean': float(alpha_opt.mean().item()),
    'oracle_scaled_mu_HR_beta_mean': float(beta_opt.mean().item()),
}
with open(save_path, 'w') as f:
    json.dump(save, f, indent=2)
print(f'\nRésultats : {save_path.absolute()}')

In [ ]:
# === Cell 11 : Visualizations ===

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# F1@p99 bar chart
variant_names = list(results.keys())
f1_vals = [results[n]['f1_p99'] for n in variant_names]
colors_f1 = ['#888' if 'baseline_only' in n or 'baseline_plus' in n
             else '#e74c3c' if 'noncausal' in n
             else '#27ae60' if 'perfect' in n or 'combined' in n
             else '#3498db' for n in variant_names]
axes[0].bar(range(len(variant_names)), f1_vals, color=colors_f1)
axes[0].axhline(F1_PASS, ls='--', color='green', alpha=0.5, label=f'PASS thresh {F1_PASS}')
axes[0].axhline(F1_FAIL, ls='--', color='red', alpha=0.5, label=f'FAIL thresh {F1_FAIL}')
axes[0].axhline(F1_NONCAUSAL_V4, ls='--', color='orange', alpha=0.7, label=f'v4 baseline {F1_NONCAUSAL_V4}')
axes[0].set_xticks(range(len(variant_names)))
axes[0].set_xticklabels(variant_names, rotation=45, ha='right', fontsize=7)
axes[0].set_ylabel('F1@p99')
axes[0].set_title('F1@p99 par variant — verdict ORACLE')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

# Pearson bar chart
pearson_vals = [results[n]['pearson'] for n in variant_names]
axes[1].bar(range(len(variant_names)), pearson_vals, color=colors_f1)
axes[1].axhline(PEARSON_NONCAUSAL_V4, ls='--', color='orange', alpha=0.7, label=f'v4 baseline {PEARSON_NONCAUSAL_V4}')
axes[1].set_xticks(range(len(variant_names)))
axes[1].set_xticklabels(variant_names, rotation=45, ha='right', fontsize=7)
axes[1].set_ylabel('Pearson')
axes[1].set_title('Pearson par variant')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

# RMSE bar chart
rmse_vals = [results[n]['rmse'] for n in variant_names]
axes[2].bar(range(len(variant_names)), rmse_vals, color=colors_f1)
axes[2].axhline(RMSE_NONCAUSAL_V4, ls='--', color='orange', alpha=0.7, label=f'v4 baseline {RMSE_NONCAUSAL_V4}')
axes[2].set_xticks(range(len(variant_names)))
axes[2].set_xticklabels(variant_names, rotation=45, ha='right', fontsize=7)
axes[2].set_ylabel('RMSE')
axes[2].set_title('RMSE par variant')
axes[2].legend(fontsize=8); axes[2].grid(True, alpha=0.3)

plt.suptitle(f'Phase 2 ORACLE — verdict : {verdict}', fontsize=12, y=1.02)
plt.tight_layout()
viz_path = OUTPUT_DIR / 'oracle_viz.png'
plt.savefig(viz_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved : {viz_path}')